In [1]:
# %matplotlib widget
import scipy
import numpy as np
import matplotlib.pyplot as plt
import symengine
from astropy import units as u
from astropy import constants as const
import pandas as pd
from astropy.units import Quantity
from astropy.visualization import quantity_support
from mpl_toolkits.mplot3d import Axes3D 
import mpl_toolkits.mplot3d as m3d
from math import lgamma, log, exp, comb
from SolverFunction import _find_unknown,auto_bracket,solve_quantity
quantity_support()


<astropy.visualization.units.quantity_support.<locals>.MplQuantityConverter at 0x76b1b9b0c050>

### Solver Definitions

In [2]:
def arcsec_to_parsec(p):
    
    """
    Converts parallax angle in arcseconds into distance to star in parsecs.
    
    """
    
    return p.to(u.parsec,equivalencies = u.parallax())

def parsec_to_arcsec(d):
    
    """
    Converts distance to star in parsecs into parallax angle in arcseconds.

    """
    
    return d.to(u.arcsecond, equivalencies = u.parallax())

def grav_accel(g,M,r):
    """
    Relationship between gravitational acceleration, mass of celestial body, and distance from center of mass as a constraint for solver.
    g (units of meters / second ^ 2): Gravitational acceleration at distance r.
    M (units of kilograms): Mass of celestial body exerting the gravitational force.
    r (units of meters): Distance from the center of mass.
    """

    return const.G * M / r ** 2 - g



def angular_separation(d,alpha,a):
    
    """
    Relationship between angular separation, distance, and physical separation as a constraint for solver.
    d (units of pc): Distance to the objects from an observer in parsecs
    alpha (arcseconds or radians): Angular separation between the objects
    a (units of pc): Distance between the two objects in parsecs
    
    """
    
    return (d * alpha.to(u.rad, equivalencies= u.parallax())).to(u.pc, u.dimensionless_angles()) - a


def distance_modulus(M,m,d):
    
    """
    Relationship between absolute magnitude, apparent magnitude, and distance as a constraint for solver.
    M (dimensionless): Absolute magnitude of star.
    m (dimensionless): Apparent magnitude of star.
    d (units of pc): Distance to the star in parsecs.
    
    """
    
    return m - 5 * np.log10(d.to(u.parsec)/(10 * u.pc)) - M

def distance_modulus_extinction(M,m,d,k):
    
    """
    Relationship between absolute magnitude, apparent magnitude, and distance as a constraint for solver.
    M (dimensionless): Absolute magnitude of star.
    m (dimensionless): Apparent magnitude of star.
    d (units of pc): Distance to the star in parsecs.
    k (units of 1/pc): Extinction coefficient of interstellar medium.
    """
    
    return m - 5 * np.log10(d.to(u.parsec) / (10 * u.pc)) - k*d - M


def Kepler1_m1m2(r,L,m1,m2,e,theta):
    
    """
    Relationship between distance from center of mass, angular momentum, mass of binary components, eccentricity, and angle in orbit as a constraint for solver.
    r (meters): distance from center of mass
    L (kilogram * meter^2 / second): angular momentum of orbiting body.
    m1 and m2 (kilograms): Masses of members of binary system.
    e (dimensionless): eccentricity of orbit.
    theta (radians): angle around orbit.
    
    """
    
    return ( L ** 2 / ( ( m1 + m2 ) / ( m1 * m2 ) ) ** 2 ) / (const.G * (m1 + m2) * ( 1 + e * np.cos(theta) )) - r

def Kepler1_totmass(r,L,mu,M,e,theta):
    
    """
    Relationship between distance from center of mass, angular momentum, total mass of binary, eccentricity, and angle in orbit as a constraint for solver.
    r (meters): distance from center of mass
    L (kilograms * meters^2 / second): angular momentum of orbiting body.
    M (kilograms): total mass of binary system
    e (dimensionless): eccentricity of orbit.
    theta (radians): angle around orbit.
    
    """
    
    return ( L ** 2 / mu ** 2 ) / (const.G * M * ( 1 + e * np.cos(theta) )) - r

def Kepler2(v,m1,m2,r,a):
    
    """
    Relationship between orbital velocity, mass of binary system, distance from center of mass, and semimajor axis length as a constraint for solver.
    v (meters/second): orbital velocity of component.
    m1 and m2 (kilograms): Masses of members of binary system.
    r (meters): Distance to the center of mass.
    a (meters): semimajor axis length.
    
    """
    
    return np.sqrt( const.G * ( m1 + m2 ) * (2 / r - 1 / a) ) - v

def Kepler3_totMass(P,M,a):
    
    """
    Relationship between orbital period, orbital radius, and total mass of system as a constraint for solver.
    P (time): Orbital period of system.
    M (mass): Total mass of system.
    a (distance): Average orbital radius of system (assume circular orbits.)
    
    """
    
    return (4 * np.pi ** 2) / (const.G * M) * a ** 3 - P ** 2

def Kepler3_m1m2(P,m1,m2,a):
    
    """
    Relationship between orbital period, orbital radius, and mass of binary components as a constraint for solver.
    P (time): Orbital period of system.
    m1 and m2 (kilograms): Masses of members of binary system.
    a (distance): Average orbital radius of system (assume circular orbits.)
    
    """
    
    return (4 * np.pi ** 2) / (const.G * (m1 + m2)) * a ** 3 - P ** 2

def radial_velocity(K,v,i):
    
    """
    Relationship between radial velocity, orbital velocity, and angle of inclination for orbit as a constraint for solver.
    K (meters/second): radial velocity of orbiting body with respect to observer.
    v (meters/second): orbital velocity of orbiting body.
    i (degrees): angle of inclination for orbit with respect to observer.
    
    """
    
    return v * np.sin(i) - K

def flux_eq(F,L,r):
    
    """
    Relationship between flux, luminosity, and distance to a star as a constraint for solver.
    F (Watts / meters ^ 2): Flux of the star.
    L (Watts): Luminosity of the star.
    r (meters): Distance from the star to the observer.
    
    """
    
    return L / (4 * np.pi * r ** 2) - F

def flux_ratio(F1,F2,m1,m2):
    
    """
    Relationship between the flux and magnitude ratios of two stars as a constraint for solver.
    F1 (W / m^2): Flux of star 1.
    F2 (W / m^2): Flux of star 2.
    m1 (dimensionless): Apparent magnitude of star 1.
    m2 (dimensionless): Apparent magnitude of star 2.
    
    """

    return -2.5 * np.log10(F1/F2) - (m1 - m2)

def Weinn_displacement(lambda_max,T):
    """
    Relationship between maximum wavelength and temperature, assuming a blackbody spectrum as a constraint for solver.
    lambda_max (meters): Peak wavelength of blackbody spectrum.
    T (Kelvin): Surface temperature of blackbody.
    
    """
    return lambda_max * T - 0.002897755 * u.m * u.K

def StefanBoltzmann(L,R,Teff):
    """
    Relationship between luminosity, radius, and effective temperature of a star as a constraint for solver.
    L (Watts): Luminosity of the star.
    R (Meters): Radius of the star.
    Teff (Kelvin): Effective Temperature of the star.
    
    """
    return 4 * np.pi * R ** 2 * const.sigma_sb * Teff**4 - L

def surface_flux(F,Teff):
    """
    Relationship between surface flux and effective temperature of a star as a constraint for solver.
    F (Watts / meters ^2): Surface flux of star.
    Teff (Kelvin): Effective Temperature of the star.
    
    """
    return Teff ** 4 * const.sigma_sb - F

def EddingtonLimit(L,kappa,M):
    """
    Relationship between mass, opacity, and Eddington Luminosity as a constraint for solver.
    L (Watts): Eddington Limit Luminosity
    kappa (meters^2 / kilogram): Opacity of the star, defined as 0.2(1+X) generally.
    M (kilograms): Mass of the star.

    """
    return (4 * np.pi * const.G * const.c / kappa * M - L).to(u.W)

def virial_theorem(E,M,R):
    """
    Relationship between the total mechanical energy of a star, the mass, and the radius of the star as a constraint for solver.
    E (Joules): Total mechanical energy available.
    M (kilograms): Mass of the star.
    R (meters): Radius of the star.

    """
    return (-3/10 * const.G * M ** 2 / R - E).to(u.J)

def mass_to_energy(M):
    """
    Converts mass into energy equivalent.

    """
    return M * const.c ** 2

def energy_to_mass(E):
    """
    Converts energy into mass equivalent.

    """
    return E / const.c ** 2

def spherical_volume(R):
    """
    Calculates the volume of a sphere given a radius.
    """
    return 4/3 * R**3 * np.pi

def ComputeMDP99_rate(src_rate, bkg_rate, duration, average_mu=0.3):
    mdp99 = 4.29/average_mu * np.sqrt(src_rate+bkg_rate)/(src_rate * np.sqrt(duration)) * 100
    return mdp99

def ComputeMDP99_counts(src_counts, bkg_counts, average_mu=0.3):
    mdp99 = 4.29/average_mu * np.sqrt(src_counts+bkg_counts)/(src_counts) * 100
    return mdp99

def calculate_pressure(altitude, p0=101325*u.Pa, T=293*u.K):
    """
    Calculates barometric pressure at a given altitude.
    Uses approximation: constant gravitational acceleration and temperature.
    """
    # Mean molar mass of Earth's atmosphere
    M = 0.0289644 * u.kg / u.mol 
    
    # Barometric formula: P = P0 * exp(-Mgz / RT)
    exponent = (-M * const.g0 * altitude) / (const.R * T)
    pressure = p0 * np.exp(exponent.decompose())
    
    return pressure.to(u.Pa)/const.atm

def Lum__density_thindisk(L,R,z):
    """
    Computes the luminosity density of the thin disk of the Milky Way as a function of galactic radius and galactic height.
    L (Solar Lum / pc**3): Luminosity Density.
    R (pc): Galactic Radius 
    z (pc): Height above disk.
    """
    L0 = 0.05 * u.L_sun * u.pc**-3
    h_r = 3.5 * u.kpc
    h_z = 0.3 * u.kpc
    return L0 * np.exp(-R/h_r) * (2/(np.exp(-z/(2*h_z)) + np.exp(z/(2*h_z)))) - L

def number_density_thindisk(n,R,z):
    """
    Computes the number density of stars within the thin disk of the Milky Way as a function of galactic radius and galactic height.
    n (1/pc**3): number of stars within a certain magnitude range.
    R (pc): Galactic Radius 
    z (pc): Height above disk.
    """
    n0 = 0.05 * u.pc**-3
    h_r = 3.5 * u.kpc
    h_z = 0.3 * u.kpc
    zthin = 2 * h_z
    return n0 * (np.exp(-z/zthin) + 0.085 * np.exp(-z/(0.33*u.kpc))) * np.exp(-R/h_r) - n

def roche_density(rho,M,d):
    """
    Computes the Roche density, which is the minimum density an object must have to avoid being tidally disrupted by a more massive body. Formatted for solver.
    M (kilograms): Mass of the more massive body.
    d (meters): Distance from the center of the more massive body.
    rho (kilograms/m^3): Density of the object.
    """
    return (3/(2*np.pi)) * (M/d**3) - rho

def tidal_force(F,M,d,m):
    """
    Computes the tidal force experienced by an object due to a more massive body. Formatted for solver.
    F (Newtons): Differential force experienced by the object due to tidal effects.
    M (kilograms): Mass of the more massive body.
    d (meters): Distance from the center of the more massive body.
    m (kilograms): Mass of the object experiencing the tidal force.
    """
    return 2 * const.G * M * m / d**3 - F

def Saha_Equation(temp,electron_pressure, charge1, charge2, ionization_energy):
    """
    Calculate the ratio of number densities of two ionization states using the Saha equation.

    Parameters:
    temp (float): Temperature in Kelvin.
    electron_pressure (float): Electron pressure in N/cm^2.
    charge1 (int): Charge of the lower ionization state.
    charge2 (int): Charge of the higher ionization state.
    ionization_energy (float): Ionization energy in eV.

    Returns:
    float: Ratio of number densities n2/n1.
    """
    # Convert ionization energy from eV to erg

    # Saha equation
    saha_const = (2 * np.pi * const.m_e * const.k_B * temp / const.h**2)**(3/2)
    exponent = -ionization_energy / (const.k_B.to(u.eV / u.K) * temp)
    ratio = saha_const * np.exp(exponent) * (2 * const.k_B * temp * charge1 / (charge2 * electron_pressure))
    return ratio.decompose()


In [3]:
def logmult(N,q):
    return N * np.log(q/N)+N+N**2/q

def normal_mult(N,q):
    return (((q+N)/q)**q*((q+N)/N)**N)/(np.sqrt(2 * np.pi*q*(q+N)/N))

try: normal_mult(10**23,100)
except: logmult(10*23,100)


In [8]:
solve_quantity(grav_accel, g=None, M=5.972e24 * u.kg, r=6.371e6 * u.m,bracket=(0.1, 20) * u.m / u.s**2)

# REMEMBER. PROPER FORM OF solve_quantity is
# solve_quantity(function, unknown_variable=None, **known_variables, bracket=(lower_bound, upper_bound)*units)


<Quantity 9.81997343 m / s2>

### Testing

In [41]:

#def wave_probability(psi,)
prob_array = []
k_size=[]
#for k in range(1,1000):
k=7
psi = lambda x: np.exp(-x**2)


def wave_prob(psi,bounds,x_range):

    """
    Finds the probability value of a wave function within the bounds.
    psi: Wave function that we wish to find the probability of.
    bounds: 2 Element list of upper and lower x-limits for evaluation.
    x_range: Defines the range of x values.
    """
    # 1. Define a wave function (e.g., Gaussian wave packet)
    lower,upper = bounds
    x = np.linspace(-x_range, x_range, 100*x_range)
    #psi = lambda x: np.exp(-x**2)#np.exp(-x**2)

    # 2. Normalize the wave function
    norm = np.sqrt(np.trapezoid(np.abs(psi(x))**2, x))
    psi_normalized = psi(x) / norm

    # 3. Calculate Probability Density |psi|^2
    probability_density = np.abs(psi_normalized)**2

    # 4. Find probability in a range (e.g., between -1 and 1)
    mask = (x > lower) & (x < upper)
    prob_in_range = np.trapezoid(probability_density[mask], x[mask])
    print(f"Probability in range: {prob_in_range:.4f}")
    return prob_in_range

wave_prob(psi,[-0.1,0.1],x_range=50)


Probability in range: 0.1429


np.float64(0.14285684543934454)

In [42]:
# Energy stored in a magnetic field:
# W = 0.5 * L * I**2
# W = 0.5*mu_0 * int(-inf,inf, B**2)*d_tau, which is in energy per unit volume.


# Example: A long coaxial cable carries current I (flows up on the inner cable, and down on the outer cable)
# -z direction on outer cable, +z direction on inner cable. Find the magnetic energy energy store in a segment of length l

# First calculate the magnetic field outside the coaxial cable. Assume radii a and b for the inner and outer cables. Segment l will be smaller than the original cable
# which we will assume extends to infinity.

# Expression of magnetic field between the two cylinders.
# This is due to Ampere's law.
B_ab = lambda s: const.mu0 * I / (2 * np.pi * s)

# We draw an Amperian Loop between the inner and outer cable to find the magnetic field between them. If we want to find the magnetic field of just the inner, ensure s<a. 
# For outside both cables, s>b

# We want to compute the inductance, which is:
# Phi = L * I
# Phi = int(-inf,inf, B)*da

# Intuitively, the magnetic energy should rely on the relationship between the two radii, where the magnetic energy should decrease as the difference between the two radii increases.
# Since the magnetic field when s<a and s>b, the bounds of our earlier integral become integration from a to b.

# The volume of the cylindrical shell of radius s, length l, and thickness ds is: volume of b - volume of a.
# V = l * np.pi * (s + ds)**2 - l * np.pi * s**2
# V = l * np.pi * 2 * ds * s = d_tau

# So our earlier integral becomes 
# W = 0.5 * const.mu0 * int of (B ** 2 * d_tau)
# W = const.mu0**2 * I**2 / 4 *np.pi**2 * int(1/s**2 * l * 2 * np.pi * s * ds)
# W = const.mu0**2 * I ** 2 * l / ( 2 * np.pi) * int_a_to_b(ds/s)

# Simplifying gives our final expression for the energy.
# W = const.mu0 * 0.5 * l / (2 * np.pi) * ln(b/a) * I**2
# Since this is in the same form as 0.5 * L * I**2, we can determine L.

# So the magnetic energy per unit volume between two coaxial cables carrying current I is:

def B_field_energy(W,L,I):
    return L * I ** 2 - W

3000 * u.J / ( 300 * u.A)

<Quantity 10. J / A>

In [ ]:
# Find the self-inductance of a toroidal coil with rectangular cross section. Total of N turns. Verify by calculating the energy.
# A toroid is a donut.
# The magnetic field B is given by:
# B = const.mu0*N*I/(2 * np.pi * s) * phi direction.
# The self-inductance is:
# Phi = L*I = int(B)*da

# Long story short, L = const.mu0 * N ** 2 * h / ( 2 * np.pi) * ln(b/a), where h is the height of the toroid, and a and b are the inner andn outer radii respectively. N is the number of turns.